# SmearDx — blood smear cell counting

Runs the trained YOLO detector from **BloodCellCountFromSmear** in Google Colab.

Upload a smear and get **RBC / WBC / Platelet** counts with bounding boxes.

The Next.js website (`http://localhost:3000`) is separate and does not run here.

## How to run

1. **Runtime → Change runtime type → GPU (T4)**
2. Run the cells in order
3. When asked, upload `SmearDx_backend.zip` (same folder as this notebook on your Mac)
4. Then upload a smear image, or use a test image from the zip

## 1. Install packages

In [ ]:
%pip install -q ultralytics pillow opencv-python-headless

## 2. Upload and unzip the project

Run this cell and choose **`SmearDx_backend.zip`**.

If the zip is already on Google Drive, set `DRIVE_ZIP` to that path and skip the file picker.

In [ ]:
from __future__ import annotations

import sys
import zipfile
from pathlib import Path

from google.colab import files

# Set this if the zip already lives on Drive, e.g. "/content/drive/MyDrive/SmearDx_backend.zip"
DRIVE_ZIP = ""
EXTRACT_DIR = Path("/content/smeardx")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

zip_path = Path(DRIVE_ZIP) if DRIVE_ZIP else None
if zip_path is None or not zip_path.is_file():
    print("Upload SmearDx_backend.zip")
    uploaded = files.upload()
    zips = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zips:
        raise FileNotFoundError("No .zip uploaded. Choose SmearDx_backend.zip.")
    zip_path = Path("/content") / zips[0]

print(f"Unzipping {zip_path} → {EXTRACT_DIR}")
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)


def find_project_dir(root: Path) -> Path:
    matches = list(root.rglob("eval_config.py"))
    if not matches:
        raise FileNotFoundError(
            "eval_config.py not found after unzip. Is this SmearDx_backend.zip?"
        )
    return matches[0].resolve().parent


PROJECT_DIR = find_project_dir(EXTRACT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project:", PROJECT_DIR)

## 3. Load the trained model

Uses the same weights and gates as the app:

- Weights: `runs/detect/runs/yolo26/full_training_from_corrected-2/weights/best.pt`
- Classes: `{0: Platelets, 1: RBC, 2: WBC}`
- Gates: RBC ≥ 0.60, WBC ≥ 0.40, Platelets ≥ 0.40

In [ ]:
from ultralytics import YOLO

from eval_config import (
    CANONICAL_CLASS_NAMES,
    CLASS_COLORS_RGB,
    CLASS_THRESHOLDS,
    DISPLAY_CLASS_ORDER,
    IMAGE_SIZE,
    MIN_INFERENCE_CONF,
    accept_detection,
    assert_canonical_mapping,
    empty_class_counts,
    resolve_final_model,
)

model_path = resolve_final_model(PROJECT_DIR)
print("Weights:", model_path)

model = YOLO(str(model_path))
assert_canonical_mapping(model.names)

print("Classes:", CANONICAL_CLASS_NAMES)
print("Gates:")
for class_name in DISPLAY_CLASS_ORDER:
    print(f"  {class_name}: {CLASS_THRESHOLDS[class_name]:.2f}")
print("Infer floor (conf):", MIN_INFERENCE_CONF)
print("Image size:", IMAGE_SIZE)

## 4. Count cells on a smear

Set `USE_TEST_IMAGE = True` to run a bundled test smear, or `False` to upload your own image.

In [ ]:
from collections import Counter

from google.colab import files
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

USE_TEST_IMAGE = True


def pick_image() -> Path:
    if USE_TEST_IMAGE:
        test_dir = PROJECT_DIR / "DataSet" / "test" / "images"
        images = sorted(
            [
                path
                for path in test_dir.iterdir()
                if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            ]
        )
        if not images:
            raise FileNotFoundError(f"No test images in {test_dir}")
        print("Using test image:", images[0].name)
        return images[0]

    print("Upload a smear image (.jpg / .png / .bmp)")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No image uploaded.")
    name = next(iter(uploaded))
    path = Path("/content") / name
    path.write_bytes(uploaded[name])
    return path


def annotate(image: Image.Image, detections: list[dict]) -> Image.Image:
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 14)
    except OSError:
        font = ImageFont.load_default()

    for detection in detections:
        class_name = detection["class"]
        confidence = detection["confidence"]
        x1, y1, x2, y2 = detection["box"]
        color = CLASS_COLORS_RGB.get(class_name, (148, 163, 184))
        draw.rectangle([int(x1), int(y1), int(x2), int(y2)], outline=color, width=3)
        label = f"{class_name} {confidence:.2f}"
        text_x, text_y = int(x1), max(0, int(y1) - 18)
        try:
            bbox = draw.textbbox((text_x, text_y), label, font=font)
            draw.rectangle(bbox, fill=color)
        except Exception:
            pass
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=font)
    return annotated


image_path = pick_image()
image = Image.open(image_path).convert("RGB")

results = model.predict(
    source=image,
    imgsz=IMAGE_SIZE,
    conf=MIN_INFERENCE_CONF,
    verbose=False,
)
result = results[0]

raw: list[dict] = []
if result.boxes is not None and len(result.boxes) > 0:
    for index in range(len(result.boxes)):
        class_id = int(result.boxes.cls[index])
        confidence = float(result.boxes.conf[index])
        class_name = CANONICAL_CLASS_NAMES.get(class_id, str(model.names[class_id]))
        raw.append(
            {
                "class": class_name,
                "confidence": round(confidence, 4),
                "box": [float(value) for value in result.boxes.xyxy[index].tolist()],
            }
        )

accepted = [
    detection
    for detection in raw
    if accept_detection(detection["class"], detection["confidence"])
]
counts = empty_class_counts()
for detection in accepted:
    if detection["class"] in counts:
        counts[detection["class"]] += 1

print("\nCounts (class gates applied)")
for class_name in DISPLAY_CLASS_ORDER:
    print(f"  {class_name}: {counts[class_name]}")
print(f"  Total: {sum(counts.values())}")
print(f"Raw boxes at conf>={MIN_INFERENCE_CONF:.2f}: {len(raw)}")
print(f"Accepted boxes: {len(accepted)}")

annotated = annotate(image, accepted)
print("\nOriginal")
display(image)
print("Annotated")
display(annotated)

print("\nPer-class accepted confidences")
by_class = Counter(detection["class"] for detection in accepted)
for class_name in DISPLAY_CLASS_ORDER:
    confs = [
        detection["confidence"]
        for detection in accepted
        if detection["class"] == class_name
    ]
    if confs:
        print(
            f"  {class_name}: n={by_class[class_name]}  "
            f"min={min(confs):.2f}  max={max(confs):.2f}"
        )
    else:
        print(f"  {class_name}: n=0")